# Task 4 - CNN Pipeline
Converted from Task_3/CNN.py to a Jupyter notebook.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.errors import NotGeoreferencedWarning

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms

In [2]:
warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

# Paths
TASK_DIR = Path.cwd()
BASE_DIR = TASK_DIR.parent
SAMPLES_DIR = BASE_DIR / "samples"
LABELS_DIR = BASE_DIR / "labels"
OUTPUT_DIR = TASK_DIR / "outputs"

# Simple settings
IMAGE_SIZE = 64
BATCH_SIZE = 16
EPOCHS = 15
LEARNING_RATE = 0.001
PATIENCE = 10
SEED = 42

# NDVI thresholds for 3 classes
LOW_THRESHOLD = 150
MEDIUM_THRESHOLD = 180
CLASS_NAMES = ["low", "medium", "high"]

np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def label_to_class(label_tile):
    mean_ndvi = float(np.nanmean(label_tile))

    if mean_ndvi < LOW_THRESHOLD:
        return 0
    if mean_ndvi < MEDIUM_THRESHOLD:
        return 1
    return 2


# -----------------------------------------------------------------------------
# Load the TIFF data
# -----------------------------------------------------------------------------

images = []
labels = []

for sample_path in sorted(SAMPLES_DIR.glob("*.tif*")):
    label_path = LABELS_DIR / sample_path.name.replace("img_", "ndvi_")

    if not label_path.exists():
        continue

    with rasterio.open(sample_path) as src:
        image = src.read(
            out_shape=(src.count, IMAGE_SIZE, IMAGE_SIZE),
            resampling=Resampling.bilinear,
        ).astype(np.float32)

    with rasterio.open(label_path) as src:
        label = src.read(1).astype(np.float32)

    image = image / 255.0
    image = np.nan_to_num(image)

    images.append(image)
    labels.append(label_to_class(label))

images = np.array(images, dtype=np.float32)
labels = np.array(labels, dtype=np.int64)

if len(images) == 0:
    raise RuntimeError("No image/label pairs were loaded.")

print(f"Total images: {len(images)}")
print(f"Class counts: {dict(zip(CLASS_NAMES, np.bincount(labels, minlength=3)))}")

Total images: 614
Class counts: {'low': np.int64(27), 'medium': np.int64(338), 'high': np.int64(249)}


In [4]:
# -----------------------------------------------------------------------------
# Split into train / validation / test
# -----------------------------------------------------------------------------

train_images, temp_images, train_labels, temp_labels = train_test_split(
    images,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=SEED,
)

val_images, test_images, val_labels, test_labels = train_test_split(
    temp_images,
    temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=SEED,
)

print(f"Train: {len(train_images)}")
print(f"Validation: {len(val_images)}")
print(f"Test: {len(test_images)}")

# Normalize using only the training set
mean = train_images.mean(axis=(0, 2, 3), keepdims=True)
std = train_images.std(axis=(0, 2, 3), keepdims=True)
std[std == 0] = 1.0

train_images = (train_images - mean) / std
val_images = (val_images - mean) / std
test_images = (test_images - mean) / std

Train: 429
Validation: 92
Test: 93


In [5]:
# -----------------------------------------------------------------------------
# TODO Data Augmentation (only for training) with pytorch 
# -----------------------------------------------------------------------------

train_transforms = transforms.Compose([
    # transforms.RandomRotation(5),
    transforms.RandomHorizontalFlip(p=0.5),
    # transforms.RandomVerticalFlip(),
    # transforms.ColorJitter(brightness=0.05, contrast=0.05),
    # transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
])

In [6]:
# Custom Dataset class to apply transforms
class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label


# Convert to PyTorch datasets with augmentation
train_data = AugmentedDataset(train_images, train_labels, transform=train_transforms)
val_data = AugmentedDataset(val_images, val_labels, transform=None)
test_data = AugmentedDataset(test_images, test_labels, transform=None)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)


# -----------------------------------------------------------------------------
# CNN model
# -----------------------------------------------------------------------------

class VegetationCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(64, 32)
        self.dropout = nn.Dropout(p=0.2)
        self.fc2 = nn.Linear(32, 3)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


net = VegetationCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

In [7]:
# Verify augmentation is active on training samples across multiple trials
num_trials = 10
non_zero_orig_aug = 0
non_zero_aug_aug = 0

for i in range(num_trials):
    orig = torch.tensor(train_images[0], dtype=torch.float32)
    aug1, _ = train_data[0]
    aug2, _ = train_data[0]

    diff_orig_aug1 = (orig - aug1).abs().mean().item()
    diff_aug1_aug2 = (aug1 - aug2).abs().mean().item()

    if diff_orig_aug1 > 0:
        non_zero_orig_aug += 1
    if diff_aug1_aug2 > 0:
        non_zero_aug_aug += 1

    print(f"Trial {i+1:02d}: orig->aug1={diff_orig_aug1:.6f}, aug1->aug2={diff_aug1_aug2:.6f}")

print("\nSummary")
print(f"Non-zero orig->aug1 in {non_zero_orig_aug}/{num_trials} trials")
print(f"Non-zero aug1->aug2 in {non_zero_aug_aug}/{num_trials} trials")

if non_zero_orig_aug == 0 and non_zero_aug_aug == 0:
    print("Augmentation appears inactive for this sample.")
else:
    print("Augmentation is active.")

Trial 01: orig->aug1=0.000000, aug1->aug2=0.000000
Trial 02: orig->aug1=0.335406, aug1->aug2=0.000000
Trial 03: orig->aug1=0.335406, aug1->aug2=0.000000
Trial 04: orig->aug1=0.335406, aug1->aug2=0.000000
Trial 05: orig->aug1=0.000000, aug1->aug2=0.000000
Trial 06: orig->aug1=0.000000, aug1->aug2=0.000000
Trial 07: orig->aug1=0.335406, aug1->aug2=0.000000
Trial 08: orig->aug1=0.335406, aug1->aug2=0.335406
Trial 09: orig->aug1=0.000000, aug1->aug2=0.000000
Trial 10: orig->aug1=0.000000, aug1->aug2=0.000000

Summary
Non-zero orig->aug1 in 5/10 trials
Non-zero aug1->aug2 in 1/10 trials
Augmentation is active.


In [8]:
# -----------------------------------------------------------------------------
# Train
# -----------------------------------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_model_path = OUTPUT_DIR / "basic_cnn.pth"

best_val_loss = float("inf")
wait = 0

for epoch in range(EPOCHS):
    net.train()
    running_loss = 0.0

    for batch_images, batch_labels in train_loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()

        outputs = net(batch_images)
        loss = loss_fn(outputs, batch_labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    net.eval()
    val_loss_total = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_images, batch_labels in val_loader:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)

            outputs = net(batch_images)
            loss = loss_fn(outputs, batch_labels)
            val_loss_total += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()

    val_loss = val_loss_total / len(val_loader)
    val_accuracy = correct / total

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        wait = 0
        torch.save(net.state_dict(), best_model_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping.")
            break

Epoch 01/15 | Train Loss: 0.8153 | Val Loss: 0.7498 | Val Acc: 0.6739
Epoch 02/15 | Train Loss: 0.6406 | Val Loss: 0.7044 | Val Acc: 0.6630
Epoch 03/15 | Train Loss: 0.6038 | Val Loss: 0.7355 | Val Acc: 0.6848
Epoch 04/15 | Train Loss: 0.5714 | Val Loss: 0.6515 | Val Acc: 0.7283
Epoch 05/15 | Train Loss: 0.5079 | Val Loss: 0.6281 | Val Acc: 0.6848
Epoch 06/15 | Train Loss: 0.4962 | Val Loss: 0.5980 | Val Acc: 0.7283
Epoch 07/15 | Train Loss: 0.4508 | Val Loss: 0.8435 | Val Acc: 0.6522
Epoch 08/15 | Train Loss: 0.4801 | Val Loss: 0.6959 | Val Acc: 0.7283
Epoch 09/15 | Train Loss: 0.4404 | Val Loss: 0.7221 | Val Acc: 0.7174
Epoch 10/15 | Train Loss: 0.4336 | Val Loss: 0.6157 | Val Acc: 0.7174
Epoch 11/15 | Train Loss: 0.4122 | Val Loss: 0.6768 | Val Acc: 0.7283
Epoch 12/15 | Train Loss: 0.4460 | Val Loss: 0.8001 | Val Acc: 0.6522
Epoch 13/15 | Train Loss: 0.4078 | Val Loss: 0.6635 | Val Acc: 0.7065
Epoch 14/15 | Train Loss: 0.3975 | Val Loss: 0.5828 | Val Acc: 0.7174
Epoch 15/15 | Train 

In [9]:
# -----------------------------------------------------------------------------
# Test
# -----------------------------------------------------------------------------

net = VegetationCNN().to(device)
net.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
net.eval()

all_true = []
all_pred = []

with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        batch_images = batch_images.to(device)

        outputs = net(batch_images)
        _, predicted = torch.max(outputs, 1)

        all_true.extend(batch_labels.numpy())
        all_pred.extend(predicted.cpu().numpy())

accuracy = accuracy_score(all_true, all_pred)
precision = precision_score(all_true, all_pred, average="macro", zero_division=0)
recall = recall_score(all_true, all_pred, average="macro", zero_division=0)
f1 = f1_score(all_true, all_pred, average="macro", zero_division=0)

print("\nTest Results")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

with open(OUTPUT_DIR / "basic_results.txt", "w") as file:
    file.write(f"Accuracy : {accuracy:.4f}\n")
    file.write(f"Precision: {precision:.4f}\n")
    file.write(f"Recall   : {recall:.4f}\n")
    file.write(f"F1-Score : {f1:.4f}\n")

print(f"\nSaved model: {best_model_path}")




Test Results
Accuracy : 0.8172
Precision: 0.8816
Recall   : 0.6563
F1-Score : 0.6860

Saved model: c:\Users\Public\OpenFrameworks\apps\myApps\PandaHat\Learning_Path\Task_4\outputs\basic_cnn.pth


## Test Results Summary

### Best Result
- **Horizontal flip (0.5) + Batch Normalization + Extended Patience**
  - Accuracy: 0.8172
  - Precision: 0.8816
  - Recall: 0.6563
  - F1-Score: 0.6860

### Results Ordered (Best to Worst)

| Configuration | Accuracy | Precision | Recall | F1-Score |
|---|---|---|---|---|
| Horizontal flip (0.5) + BatchNorm + Patience | 0.8172 | 0.8816 | 0.6563 | 0.6860 |
| Horizontal flip (1.0) + BatchNorm + Patience | 0.7957 | 0.8643 | 0.6387 | 0.6713 |
| Horizontal flip only | 0.6989 | 0.4676 | 0.4919 | 0.4756 |
| Horizontal flip + Rotation | 0.6989 | 0.4648 | 0.4897 | 0.4754 |
| No Augmentation | 0.6882 | 0.4557 | 0.4742 | 0.4646 |
| Horizontal flip + Rotation + Color Jitter | 0.6344 | 0.4300 | 0.4214 | 0.4109 |
| Heavy Augmentation (all transforms) | 0.5484 | 0.1828 | 0.3333 | 0.2361 |


### Key Observations
- **Too much augmentation hurts performance** - The combination of rotation, vertical flip, color jitter, and random crop reduced accuracy significantly
- **Moderate horizontal flip (0.5) is optimal** - Better than both no augmentation and aggressive flipping (1.0)
- **Batch normalization + patience are essential** - These significantly improved results